In [ ]:
import sympy as sp

# -----------------------------
# Define coordinates and metric
# -----------------------------

# Example: 2D chart (x, y)
x, y = sp.symbols('x y')
coords = [x, y]

# Example metric (replace with your own)
g = sp.Matrix([
    [1 + x**2, 0],
    [0, 1]
])

# Polar metric (replace with your own)
r, theta, phi = sp.symbols('r theta phi')
coords = [r, theta, phi]
g = sp.Matrix([
    [1, 0, 0],
    [0, r**2, 0],
    [0, 0, r**2 * sp.sin(theta)**2]
])

# 2d sphere metric (replace with your own)
theta, phi = sp.symbols('theta phi')
r = sp.symbols('r', positive=True, real=True)
coords = [theta, phi]

g = sp.Matrix([
    [r**2, 0],
    [0, r**2 * sp.sin(theta)**2]
])

# # 3d sphere metric (replace with your own)
# r, theta, phi = sp.symbols('r theta phi')
# coords = [r, theta, phi]

# g = sp.Matrix([
#     [1, 0, 0],
#     [0, r**2, 0],
#     [0, 0, r**2 * sp.sin(theta)**2]
# ])

# Inverse metric
g_inv = g.inv()
g

In [ ]:
# -----------------------------
# 1. Christoffel symbols
# -----------------------------
n = len(coords)
Gamma = sp.MutableDenseNDimArray.zeros(n, n, n)  # Gamma^i_{jk}

for i in range(n):
    for j in range(n):
        for k in range(n):
            term = 0
            for l in range(n):
                term += g_inv[i, l] * (
                    sp.diff(g[l, j], coords[k]) +
                    sp.diff(g[l, k], coords[j]) -
                    sp.diff(g[j, k], coords[l])
                )
            Gamma[i, j, k] = sp.nsimplify(0.5 * term, rational=True)
Gamma

In [ ]:
# -----------------------------
# 2. Riemann curvature tensor
# -----------------------------
Riemann = sp.MutableDenseNDimArray.zeros(n, n, n, n)  # R^i_{jkl}

for i in range(n):
    for j in range(n):
        for k in range(n):
            for l in range(n):
                expr = (
                    sp.diff(Gamma[i, j, l], coords[k])
                    - sp.diff(Gamma[i, j, k], coords[l])
                    + sum(Gamma[i, k, m] * Gamma[m, j, l] - Gamma[i, l, m] * Gamma[m, j, k]
                          for m in range(n))
                )
                expr = sp.simplify(expr)

                # Clean up: force zero if it's symbolically zero
                if expr == 0 or expr.is_zero:
                    expr = sp.S(0)
                else:
                    expr = sp.nsimplify(expr, rational=True)
                
                Riemann[i, j, k, l] = expr
Riemann

In [ ]:
Riemann[1,0,:,:]

In [ ]:
# -----------------------------
# 3. Ricci tensor
# -----------------------------
Ricci = sp.MutableDenseNDimArray.zeros(n, n)

for j in range(n):
    for k in range(n):
        Ricci[j, k] = sum(Riemann[i, j, i, k] for i in range(n))
        Ricci[j, k] = sp.nsimplify(Ricci[j, k])

Ricci

In [ ]:
# -----------------------------
# 4. Ricci scalar
# -----------------------------
Ricci_scalar = sum(g_inv[j, k] * Ricci[j, k] for j in range(n) for k in range(n))
Ricci_scalar = sp.nsimplify(Ricci_scalar)

Ricci_scalar